In [ ]:
# ==============================================================================
# Cell 1: Dependency Installation
# ==============================================================================
%pip install scanpy anndata scikit-misc rpy2 gdown networkx

In [ ]:
# ==============================================================================
# Cell 2: Core Architecture, Graph Engines, & Pipeline Implementation
# ==============================================================================
import os
import random
import copy
from typing import Optional, Tuple, Dict, Any, List

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
from scipy.sparse import coo_matrix, csc_matrix, csr_matrix, issparse
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from sklearn.decomposition import PCA
from sklearn.preprocessing import Normalizer, LabelEncoder
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    silhouette_score
)
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.backends import cudnn
import scanpy as sc
import anndata

# ------------------------------------------------------------------------------
# 1. Reproducibility & Utilities
# ------------------------------------------------------------------------------
def fix_seed(seed: int = 2022):
    """Ensures complete determinism across CPU/GPU operations."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

def init_weights(*modules):
    for m in modules:
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, Parameter):
            nn.init.xavier_uniform_(m)

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse.FloatTensor(indices, values, shape)

def normalize_adj(adj):
    """Symmetric GCN graph normalization: D^(-1/2) * (A + I) * D^(-1/2)"""
    adj = sp.coo_matrix(adj)
    adj_ = adj + sp.eye(adj.shape[0])
    rowsum = np.array(adj_.sum(1))
    degree_mat_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    adj_normalized = adj_.dot(degree_mat_inv_sqrt).transpose().dot(degree_mat_inv_sqrt).tocoo()
    return adj_normalized

# ------------------------------------------------------------------------------
# 2. Data Preprocessing (Scanpy, CLR, LSI/TF-IDF)
# ------------------------------------------------------------------------------
def clr_normalize_each_cell(adata: anndata.AnnData, inplace: bool = True) -> anndata.AnnData:
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x)) if len(x) > 0 else 1.0
        return np.log1p(x / exp)
    if not inplace:
        adata = adata.copy()
    adata.X = np.apply_along_axis(
        seurat_clr, 1, (adata.X.toarray() if issparse(adata.X) else np.array(adata.X))
    )
    return adata

def tfidf(X):
    idf = X.shape[0] / (X.sum(axis=0) + 1e-12)
    if issparse(X):
        tf = X.multiply(1.0 / (X.sum(axis=1) + 1e-12))
        return tf.multiply(idf)
    else:
        tf = X / (X.sum(axis=1, keepdims=True) + 1e-12)
        return tf * idf

def lsi(adata: anndata.AnnData, n_components: int = 50, use_highly_variable: Optional[bool] = None, **kwargs):
    if use_highly_variable is None:
        use_highly_variable = "highly_variable" in adata.var
    adata_use = adata[:, adata.var["highly_variable"]] if use_highly_variable else adata
    X = tfidf(adata_use.X)
    X_norm = Normalizer(norm="l1").fit_transform(X)
    X_norm = np.log1p(X_norm * 1e4)
    X_lsi = sklearn.utils.extmath.randomized_svd(X_norm, n_components + 1, **kwargs)[0]
    X_lsi -= X_lsi.mean(axis=1, keepdims=True)
    X_lsi /= (X_lsi.std(axis=1, ddof=1, keepdims=True) + 1e-12)
    adata.obsm["X_lsi"] = X_lsi[:, 1:]

def pca(adata: anndata.AnnData, use_reps: Optional[str] = None, n_comps: int = 30) -> np.ndarray:
    pca_model = PCA(n_components=n_comps)
    if use_reps is not None:
        feat_pca = pca_model.fit_transform(adata.obsm[use_reps])
    else:
        feat_pca = pca_model.fit_transform(adata.X.toarray() if issparse(adata.X) else adata.X)
    return feat_pca

# ------------------------------------------------------------------------------
# 3. Hybrid Topology Construction Engine (ARISE + SpaFusion + SMART)
# ------------------------------------------------------------------------------
def construct_3node_motif(adj: np.ndarray) -> np.ndarray:
    """Computes normalized 3-node motif co-occurrence matrix M_3 = (A^2) \odot A [SpaFusion]."""
    adj_bin = (adj > 0).astype(np.float32)
    adj_sq = np.dot(adj_bin, adj_bin)
    m3 = adj_sq * adj_bin
    max_val = m3.max()
    if max_val > 0:
        m3 = m3 / max_val
    return m3

def construct_archer_graphs(adata_omics1: anndata.AnnData, adata_omics2: anndata.AnnData,
                            datatype: str = '10x', n_neighbors_spatial: int = 6,
                            n_neighbors_feature: int = 20) -> Dict[str, Any]:
    """
    Synthesizes:
    - Symmetrized Physical Proximity Graph (A_s)
    - RNA-Anchored Intersection Topology (A_com = A_f^RNA \cap A_s) [ARISE]
    - Refined Spatial Graph (A_s^v = A^v \odot \hat{A}_s) [SpaFusion]
    - High-Order 3-Node Motif Graph (A_motif^v = 0.5*A^v + 0.5*M_3^v) [SpaFusion]
    """
    coords = adata_omics1.obsm['spatial']
    nbrs = NearestNeighbors(n_neighbors=n_neighbors_spatial + 1).fit(coords)
    adj_s_raw = nbrs.kneighbors_graph(coords, mode='connectivity').toarray()
    adj_s_raw = np.maximum(adj_s_raw, adj_s_raw.T)
    np.fill_diagonal(adj_s_raw, 0)

    # Feature Graphs (KNN)
    f1 = adata_omics1.obsm['feat']
    f2 = adata_omics2.obsm['feat']
    adj_f1_raw = kneighbors_graph(f1, n_neighbors_feature, mode='connectivity', metric='correlation').toarray()
    adj_f2_raw = kneighbors_graph(f2, n_neighbors_feature, mode='connectivity', metric='correlation').toarray()
    adj_f1_raw = np.maximum(adj_f1_raw, adj_f1_raw.T)
    adj_f2_raw = np.maximum(adj_f2_raw, adj_f2_raw.T)
    np.fill_diagonal(adj_f1_raw, 0)
    np.fill_diagonal(adj_f2_raw, 0)

    # 1. ARISE: RNA-Anchored Shared-Edge Topology (Intersection)
    A_com = ((adj_f1_raw > 0) & (adj_s_raw > 0)).astype(np.float32)

    # 2. SpaFusion: Refined Spatial Graphs
    A_s_refined1 = ((adj_f1_raw > 0) & (adj_s_raw > 0)).astype(np.float32)
    A_s_refined2 = ((adj_f2_raw > 0) & (adj_s_raw > 0)).astype(np.float32)

    # 3. SpaFusion: High-Order 3-Node Motif Graphs
    M3_1 = construct_3node_motif(adj_f1_raw)
    M3_2 = construct_3node_motif(adj_f2_raw)
    A_motif1 = 0.5 * adj_f1_raw + 0.5 * M3_1
    A_motif2 = 0.5 * adj_f2_raw + 0.5 * M3_2

    return {
        'adj_spatial_raw': torch.FloatTensor(adj_s_raw),
        'adj_com_norm': sparse_mx_to_torch_sparse_tensor(normalize_adj(A_com)),
        'adj_s1_norm': sparse_mx_to_torch_sparse_tensor(normalize_adj(A_s_refined1)),
        'adj_s2_norm': sparse_mx_to_torch_sparse_tensor(normalize_adj(A_s_refined2)),
        'adj_m1_norm': sparse_mx_to_torch_sparse_tensor(normalize_adj(A_motif1)),
        'adj_m2_norm': sparse_mx_to_torch_sparse_tensor(normalize_adj(A_motif2)),
    }

# ------------------------------------------------------------------------------
# 4. MNN Triplet Mining Engine (SMART)
# ------------------------------------------------------------------------------
def build_mnn_triplets(features: np.ndarray, top_k: int = 3, neg_ratio: float = 0.6) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Constructs (Anchor, Positive, Semi-Hard Negative) triplets via Mutual Nearest Neighbors [SMART].
    """
    n_spots = features.shape[0]
    nbrs = NearestNeighbors(n_neighbors=top_k + 1, metric='euclidean').fit(features)
    _, indices = nbrs.kneighbors(features)
    
    # Identify MNN pairs
    anchors, positives = [], []
    for i in range(n_spots):
        for neighbor in indices[i, 1:]:
            if i in indices[neighbor, 1:]:
                anchors.append(i)
                positives.append(neighbor)
                
    if len(anchors) == 0:
        # Fallback to 1-NN if MNN set is too sparse
        anchors = list(range(n_spots))
        positives = [indices[i, 1] for i in range(n_spots)]

    # Compute distances for semi-hard negatives
    dists = scipy.spatial.distance.cdist(features, features, metric='euclidean')
    neg_threshold_idx = int(n_spots * neg_ratio)
    negatives = []
    for a in anchors:
        far_indices = np.argsort(dists[a])[neg_threshold_idx:]
        negatives.append(np.random.choice(far_indices))

    return (torch.LongTensor(anchors), torch.LongTensor(positives), torch.LongTensor(negatives))

# ------------------------------------------------------------------------------
# 5. Neural Network Modules (ARCHER Architecture)
# ------------------------------------------------------------------------------
class GNNLayer(nn.Module):
    """Graph Convolutional Layer supporting Sparse Tensor Message Passing."""
    def __init__(self, in_feat: int, out_feat: int, act=F.relu):
        super().__init__()
        self.weight = Parameter(torch.FloatTensor(in_feat, out_feat))
        self.bias = Parameter(torch.FloatTensor(out_feat))
        self.act = act
        init_weights(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        support = torch.mm(x, self.weight)
        out = torch.spmm(adj, support) + self.bias
        return self.act(out) if self.act is not None else out

class TransformerEncoder(nn.Module):
    """Global Context Encoder [SpaFusion / spaLLM review]."""
    def __init__(self, in_feat: int, out_feat: int, nhead: int = 4, num_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.input_proj = nn.Linear(in_feat, out_feat)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=out_feat, nhead=nhead, dim_feedforward=out_feat * 2,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(out_feat)
        init_weights(self.input_proj)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, D] -> [1, N, D]
        h = self.input_proj(x).unsqueeze(0)
        h = self.transformer(h).squeeze(0)
        return self.norm(h)

class IntraOmicFusion(nn.Module):
    """Intra-Omic Information Fusion with Self-Correlation Enhancement [SpaFusion]."""
    def __init__(self, dim: int):
        super().__init__()
        self.weights = Parameter(torch.FloatTensor(3))  # [a, b, c] for motif, spatial, global
        self.alpha = Parameter(torch.zeros(1))
        nn.init.constant_(self.weights, 1.0 / 3.0)

    def forward(self, z_motif: torch.Tensor, z_spatial: torch.Tensor, z_global: torch.Tensor, adj_motif: torch.Tensor) -> torch.Tensor:
        w = F.softmax(self.weights, dim=0)
        z_linear = w[0] * z_motif + w[1] * z_spatial + w[2] * z_global
        
        # Structure propagation
        l_v = torch.spmm(adj_motif, z_linear)
        
        # Self-correlation matrix
        s_mat = torch.matmul(l_v, l_v.T)
        s_norm = F.softmax(s_mat / (l_v.shape[-1] ** 0.5), dim=-1)
        h_v = torch.matmul(s_norm, l_v)
        
        z_tilde = l_v + self.alpha * h_v
        return z_tilde

class ARCHER_Model(nn.Module):
    """Complete ARCHER Neural Network Engine."""
    def __init__(self, in_dim1: int, in_dim2: int, hidden_dim: int = 64, latent_dim: int = 64, n_clusters: int = 7):
        super().__init__()
        self.in_dim1 = in_dim1
        self.in_dim2 = in_dim2
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.n_clusters = n_clusters

        # Modality 1 Encoders (RNA)
        self.enc_m1 = GNNLayer(in_dim1, hidden_dim)
        self.enc_s1 = GNNLayer(in_dim1, hidden_dim)
        self.global_enc1 = TransformerEncoder(in_dim1, hidden_dim)
        self.intra_fuse1 = IntraOmicFusion(hidden_dim)

        # Modality 2 Encoders (ADT / ATAC)
        self.enc_m2 = GNNLayer(in_dim2, hidden_dim)
        self.enc_s2 = GNNLayer(in_dim2, hidden_dim)
        self.global_enc2 = TransformerEncoder(in_dim2, hidden_dim)
        self.intra_fuse2 = IntraOmicFusion(hidden_dim)

        # Refinement MLPs
        self.mlp1 = nn.Sequential(nn.Linear(hidden_dim, latent_dim), nn.ReLU(), nn.Linear(latent_dim, latent_dim))
        self.mlp2 = nn.Sequential(nn.Linear(hidden_dim, latent_dim), nn.ReLU(), nn.Linear(latent_dim, latent_dim))

        # Modality Decoders
        self.dec1 = GNNLayer(latent_dim, in_dim1, act=None)
        self.dec2 = GNNLayer(latent_dim, in_dim2, act=None)
        self.dec_joint1 = nn.Linear(latent_dim, in_dim1)
        self.dec_joint2 = nn.Linear(latent_dim, in_dim2)

        # Cluster Centroids Parameter (Consensus Self-Training)
        self.cluster_layer = Parameter(torch.Tensor(n_clusters, latent_dim))
        torch.nn.init.xavier_uniform_(self.cluster_layer)

        init_weights(self.mlp1, self.mlp2, self.dec_joint1, self.dec_joint2)

    def encode(self, x1: torch.Tensor, x2: torch.Tensor, graphs: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Modality 1 Local + Global
        zm1 = self.enc_m1(x1, graphs['adj_m1_norm'])
        zs1 = self.enc_s1(x1, graphs['adj_s1_norm'])
        zg1 = self.global_enc1(x1)
        ztilde1 = self.intra_fuse1(zm1, zs1, zg1, graphs['adj_m1_norm'])
        z1 = self.mlp1(ztilde1)

        # Modality 2 Local + Global (Anchored on A_com for spatial stability)
        zm2 = self.enc_m2(x2, graphs['adj_m2_norm'])
        zs2 = self.enc_s2(x2, graphs['adj_com_norm'])
        zg2 = self.global_enc2(x2)
        ztilde2 = self.intra_fuse2(zm2, zs2, zg2, graphs['adj_m2_norm'])
        z2 = self.mlp2(ztilde2)

        # Variance-Adaptive Inter-Omic Aggregation [SpaFusion]
        var1 = torch.var(z1)
        var2 = torch.var(z2)
        beta1 = var1 / (var1 + var2 + 1e-12)
        beta2 = 1.0 - beta1
        z_fused = beta1 * z1 + beta2 * z2

        return z_fused, z1, z2

    def forward(self, x1: torch.Tensor, x2: torch.Tensor, graphs: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        z_fused, z1, z2 = self.encode(x1, x2, graphs)

        # Reconstructions
        rec_x1 = self.dec1(z1, graphs['adj_s1_norm'])
        rec_x2 = self.dec2(z2, graphs['adj_s2_norm'])
        rec_joint_x1 = self.dec_joint1(z_fused)
        rec_joint_x2 = self.dec_joint2(z_fused)

        return {
            'z_fused': z_fused, 'z1': z1, 'z2': z2,
            'rec_x1': rec_x1, 'rec_x2': rec_x2,
            'rec_joint_x1': rec_joint_x1, 'rec_joint_x2': rec_joint_x2
        }

    def get_soft_assignment(self, z: torch.Tensor) -> torch.Tensor:
        """Student's t-distribution soft cluster assignments [SpaFusion / DEC]."""
        q = 1.0 / (1.0 + torch.sum(torch.pow(z.unsqueeze(1) - self.cluster_layer, 2), dim=2))
        q = q.pow(1.0)
        q = (q.t() / torch.sum(q, dim=1)).t()
        return q

def target_distribution(q: torch.Tensor) -> torch.Tensor:
    """Sharpened target distribution P emphasizing high-confidence assignments."""
    p = q ** 2 / q.sum(0)
    return (p.t() / p.sum(1)).t()

# ------------------------------------------------------------------------------
# 6. Quad-Objective Loss Functions
# ------------------------------------------------------------------------------
def compute_archer_losses(model_out: Dict[str, torch.Tensor], x1: torch.Tensor, x2: torch.Tensor,
                          adj_spatial_raw: torch.Tensor, triplets: Tuple[torch.Tensor, torch.Tensor, torch.Tensor],
                          q_joint: Optional[torch.Tensor] = None, p_target: Optional[torch.Tensor] = None,
                          q1: Optional[torch.Tensor] = None, q2: Optional[torch.Tensor] = None,
                          weights: Dict[str, float] = None) -> Tuple[torch.Tensor, Dict[str, float]]:
    if weights is None:
        weights = {'recon': 1.0, 'triplet': 0.5, 'spatial': 10.0, 'consensus': 1.0, 'aux': 0.1}

    # 1. Multi-Scale Reconstruction Loss
    loss_recon = F.mse_loss(x1, model_out['rec_x1']) + 5.0 * F.mse_loss(x2, model_out['rec_x2']) + \
                 F.mse_loss(x1, model_out['rec_joint_x1']) + 5.0 * F.mse_loss(x2, model_out['rec_joint_x2'])

    # 2. Auxiliary MSE Alignment
    loss_aux = F.mse_loss(model_out['z_fused'], model_out['z1']) + F.mse_loss(model_out['z_fused'], model_out['z2'])

    # 3. SMART: MNN Metric Triplet Margin Loss
    anc, pos, neg = triplets
    z = model_out['z_fused']
    d_pos = torch.sum((z[anc] - z[pos]) ** 2, dim=1)
    d_neg = torch.sum((z[anc] - z[neg]) ** 2, dim=1)
    loss_triplet = torch.mean(F.relu(d_pos - d_neg + 0.5))

    # 4. ARISE: Spatial Coherence Regularization
    z_norm = F.normalize(z, p=2, dim=1)
    cos_sim = torch.mm(z_norm, z_norm.T)
    cos_sim_sig = torch.sigmoid(cos_sim)
    loss_spatial = -torch.mean(
        adj_spatial_raw * torch.log(cos_sim_sig + 1e-12) +
        (1.0 - adj_spatial_raw) * torch.log(1.0 - cos_sim_sig + 1e-12)
    )

    # 5. SpaFusion: Consensus Clustering KL Loss
    if q_joint is not None and p_target is not None and q1 is not None and q2 is not None:
        q_avg = (q_joint + q1 + q2) / 3.0
        loss_consensus = F.kl_div(q_avg.log(), p_target, reduction='batchmean')
    else:
        loss_consensus = torch.tensor(0.0, device=z.device)

    total_loss = weights['recon'] * loss_recon + \
                 weights['aux'] * loss_aux + \
                 weights['triplet'] * loss_triplet + \
                 weights['spatial'] * loss_spatial + \
                 weights['consensus'] * loss_consensus

    return total_loss, {
        'loss_total': total_loss.item(), 'loss_recon': loss_recon.item(),
        'loss_triplet': loss_triplet.item(), 'loss_spatial': loss_spatial.item(),
        'loss_consensus': loss_consensus.item()
    }

# ------------------------------------------------------------------------------
# 7. Trainer Pipeline Class
# ------------------------------------------------------------------------------
class Train_ARCHER:
    def __init__(self, data: Dict[str, anndata.AnnData], datatype: str = '10x',
                 device: torch.device = torch.device('cpu'), pretrain_epochs: int = 5000,
                 epochs: int = 2500, lr: float = 1e-3, n_clusters: int = 7, random_seed: int = 2022):
        self.data = data
        self.datatype = datatype
        self.device = device
        self.pretrain_epochs = pretrain_epochs
        self.epochs = epochs
        self.lr = lr
        self.n_clusters = n_clusters
        self.random_seed = random_seed

        self.adata1 = self.data['adata_omics1']
        self.adata2 = self.data['adata_omics2']

        # Construct Graphs
        n_sp = 6 if datatype in ['10x', 'Stereo-CITE-seq', 'Spatial-epigenome-transcriptome'] else 3
        self.graphs_cpu = construct_archer_graphs(self.adata1, self.adata2, datatype=datatype, n_neighbors_spatial=n_sp)
        self.graphs = {k: (v.to(self.device) if isinstance(v, torch.Tensor) else v) for k, v in self.graphs_cpu.items()}

        self.x1 = torch.FloatTensor(self.adata1.obsm['feat']).to(self.device)
        self.x2 = torch.FloatTensor(self.adata2.obsm['feat']).to(self.device)

        # Build MNN Triplets
        anc, pos, neg = build_mnn_triplets(self.adata1.obsm['feat'], top_k=3, neg_ratio=0.6)
        self.triplets = (anc.to(self.device), pos.to(self.device), neg.to(self.device))

        self.model = ARCHER_Model(self.x1.shape[1], self.x2.shape[1], hidden_dim=64,
                                  latent_dim=64, n_clusters=self.n_clusters).to(self.device)

    def train(self) -> Dict[str, Any]:
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-4)

        # Stage 1: Pre-training (Reconstruction + Triplet + Spatial Coherence)
        pbar = tqdm(range(self.pretrain_epochs), desc="[ARCHER Stage 1: Warmup]")
        for epoch in pbar:
            self.model.train()
            out = self.model(self.x1, self.x2, self.graphs)
            loss, loss_dict = compute_archer_losses(
                out, self.x1, self.x2, self.graphs['adj_spatial_raw'], self.triplets,
                weights={'recon': 1.0, 'triplet': 0.5, 'spatial': 5.0, 'consensus': 0.0, 'aux': 0.1}
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix({'Loss': f"{loss.item():.4f}", 'Recon': f"{loss_dict['loss_recon']:.4f}"})

        # Initialize Cluster Centers via K-Means on Warmup Embeddings
        self.model.eval()
        with torch.no_grad():
            z_warmup, z1_w, z2_w = self.model.encode(self.x1, self.x2, self.graphs)
            from sklearn.cluster import KMeans
            kmeans = KMeans(n_clusters=self.n_clusters, n_init=20, random_state=self.random_seed)
            y_pred = kmeans.fit_predict(z_warmup.cpu().numpy())
            self.model.cluster_layer.data.copy_(torch.FloatTensor(kmeans.cluster_centers_).to(self.device))

        # Stage 2: End-to-End Consensus Fine-Tuning
        pbar = tqdm(range(self.epochs), desc="[ARCHER Stage 2: Consensus]")
        for epoch in pbar:
            self.model.train()
            out = self.model(self.x1, self.x2, self.graphs)
            
            # Compute soft & target assignments
            q_joint = self.model.get_soft_assignment(out['z_fused'])
            q1 = self.model.get_soft_assignment(out['z1'])
            q2 = self.model.get_soft_assignment(out['z2'])
            p = target_distribution(q_joint.detach())

            loss, loss_dict = compute_archer_losses(
                out, self.x1, self.x2, self.graphs['adj_spatial_raw'], self.triplets,
                q_joint=q_joint, p_target=p, q1=q1, q2=q2,
                weights={'recon': 1.0, 'triplet': 0.5, 'spatial': 10.0, 'consensus': 1.0, 'aux': 0.1}
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix({'Total': f"{loss.item():.4f}", 'KL': f"{loss_dict['loss_consensus']:.4f}"})

        # Inference
        self.model.eval()
        with torch.no_grad():
            z_fused, z1, z2 = self.model.encode(self.x1, self.x2, self.graphs)
            q_final = self.model.get_soft_assignment(z_fused)
            y_final = torch.argmax(q_final, dim=1).cpu().numpy()

        return {
            'ARCHER_emb': F.normalize(z_fused, p=2, dim=1).cpu().numpy(),
            'z1': F.normalize(z1, p=2, dim=1).cpu().numpy(),
            'z2': F.normalize(z2, p=2, dim=1).cpu().numpy(),
            'y_pred_consensus': y_final
        }

# ------------------------------------------------------------------------------
# 8. Clustering & Mclust Evaluator
# ------------------------------------------------------------------------------
def mclust_R(adata: anndata.AnnData, num_cluster: int, modelNames: str = 'EEE',
             used_obsm: str = 'emb_pca', random_seed: int = 2020) -> anndata.AnnData:
    import rpy2.robjects as robjects
    from rpy2.robjects import pandas2ri, default_converter
    from rpy2.robjects.conversion import localconverter
    np.random.seed(random_seed)
    robjects.r.library("mclust")
    robjects.r["set.seed"](random_seed)
    rmclust = robjects.r["Mclust"]
    X = np.array(adata.obsm[used_obsm], dtype=np.float64)
    df = pd.DataFrame(X, columns=[f'PC{i+1}' for i in range(X.shape[1])])
    subset_size = min(300, X.shape[0])
    subset_indices = robjects.IntVector(list(np.random.choice(range(1, X.shape[0] + 1), subset_size, replace=False)))
    init_list = robjects.ListVector({'subset': subset_indices})
    with localconverter(default_converter + pandas2ri.converter):
        res = rmclust(df, G=num_cluster, modelNames=modelNames, initialization=init_list)
    mclust_res = np.array(res['classification'])
    adata.obs['mclust'] = mclust_res
    adata.obs['mclust'] = adata.obs['mclust'].astype('int').astype('str').astype('category')
    return adata

def clustering(adata: anndata.AnnData, n_clusters: int = 7, key: str = 'ARCHER_emb',
               add_key: str = 'ARCHER', method: str = 'mclust', use_pca: bool = True,
               n_comps: int = 20, random_seed: int = 2020):
    if use_pca:
        adata.obsm[key + '_pca'] = pca(adata, use_reps=key, n_comps=n_comps)
    if method == 'mclust':
        used = key + '_pca' if use_pca else key
        adata = mclust_R(adata, used_obsm=used, num_cluster=n_clusters, random_seed=random_seed)
        adata.obs[add_key] = adata.obs['mclust']

In [ ]:
# ==============================================================================
# Cell 3: Automated Benchmark Across 6 Datasets and Evaluation Suite
# ==============================================================================
# Setup R_HOME and rpy2 environment
os.environ['R_HOME'] = '/usr/lib/R'
os.environ['PATH'] = '/usr/lib/R/bin:' + os.environ['PATH']
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, default_converter
import rpy2.robjects.conversion as cv
cv.set_conversion(default_converter + pandas2ri.converter)
robjects.r.options(warn=-1)
robjects.r('''
if (!requireNamespace("mclust", quietly = TRUE)) {
    install.packages("mclust", repos="https://cloud.r-project.org")
}
library(mclust)
''')

choices = [
    ("10x_human_lymph_node_A1", "https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC"),
    ("10x_human_lymph_node_D1", "https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz"),
    ("Mouse_Brain_E11_S1", "https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo"),
    ("Mouse_Brain_E13_S1", "https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud"),
    ("Mouse_Brain_E15_S1", "https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD"),
    ("Mouse_Brain_E18_S1", "https://drive.google.com/drive/folders/1Xj1LNIAY93biS6JIMKNRODn5GvtCKADB")
]

DATASET_INDICES = [0, 1, 2, 3, 4, 5]
SEEDS = [42, 0, 1, 7, 123, 1234, 2022, 2023, 2024, 1337]
tool = 'mclust'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

os.makedirs("results", exist_ok=True)
all_results = []

for dataset_idx in DATASET_INDICES:
    dataset_name, folder_url = choices[dataset_idx]
    print("\n" + "#" * 80)
    print(f" STARTING BENCHMARK DATASET: {dataset_name} ".center(80, "#"))
    print("#" * 80)

    base = f"data/{dataset_name}"
    os.makedirs(base, exist_ok=True)
    rna_path = os.path.join(base, "adata_RNA.h5ad")

    if dataset_name.startswith("10x"):
        other_path = os.path.join(base, "adata_ADT.h5ad")
        annotation_path = os.path.join(base, "annotation.csv")
        gt_column = "manual-anno"
        data_type = '10x'
    else:
        other_path = os.path.join(base, "adata_ATAC.h5ad")
        annotation_path = os.path.join(base, "anno.csv")
        gt_column = "cluster"
        data_type = 'Spatial-epigenome-transcriptome'

    if not os.path.exists(rna_path) or not os.path.exists(other_path) or not os.path.exists(annotation_path):
        print(f"Downloading dataset files into: {base}")
        gdown_cmd = ".venv/bin/gdown" if os.path.exists(".venv/bin/gdown") else "gdown"
        os.system(f'{gdown_cmd} --folder "{folder_url}" --output "{base}"')
    else:
        print(f"Dataset files already exist at {base}. Skipping download.")

    adata_omics1 = sc.read_h5ad(rna_path)
    adata_omics2 = sc.read_h5ad(other_path)
    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()

    anno_df = pd.read_csv(annotation_path, index_col=0)
    adata_omics1.obs['ground_truth'] = anno_df[gt_column]
    adata_omics2.obs['ground_truth'] = anno_df[gt_column]

    sc.pp.filter_genes(adata_omics1, min_cells=10)
    if not dataset_name.startswith("10x"):
        sc.pp.filter_cells(adata_omics1, min_genes=200)

    sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_omics1, target_sum=1e4)
    sc.pp.log1p(adata_omics1)
    sc.pp.scale(adata_omics1)
    adata_omics1_high = adata_omics1[:, adata_omics1.var['highly_variable']]

    if dataset_name.startswith("10x"):
        adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=adata_omics2.n_vars - 1)
        adata_omics2 = clr_normalize_each_cell(adata_omics2)
        sc.pp.scale(adata_omics2)
        adata_omics2.obsm["feat"] = pca(adata_omics2, n_comps=adata_omics2.n_vars - 1)
    else:
        adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=50)
        adata_omics2 = adata_omics2[adata_omics1.obs_names].copy()
        if "X_lsi" not in adata_omics2.obsm:
            sc.pp.highly_variable_genes(adata_omics2, flavor="seurat_v3", n_top_genes=3000)
            lsi(adata_omics2, use_highly_variable=False, n_components=51)
        adata_omics2.obsm["feat"] = adata_omics2.obsm["X_lsi"].copy()

    n_ground_truth = adata_omics1.obs["ground_truth"].nunique()
    print(f"Number of ground truth classes: {n_ground_truth}")

    dataset_results = []
    for seed in SEEDS:
        print(f"\n" + "-" * 60)
        print(f" Dataset: {dataset_name} | Seed: {seed} ".center(60, "-"))
        print("-" * 60)

        fix_seed(seed)
        data_input = {'adata_omics1': adata_omics1.copy(), 'adata_omics2': adata_omics2.copy()}
        
        # Train ARCHER Framework
        trainer = Train_ARCHER(data_input, datatype=data_type, device=device,
                               pretrain_epochs=5000, epochs=2500, lr=1e-3,
                               n_clusters=n_ground_truth, random_seed=seed)
        output = trainer.train()

        adata = data_input['adata_omics1'].copy()
        adata.obsm['ARCHER_emb'] = output['ARCHER_emb'].copy()
        clustering(adata, key='ARCHER_emb', add_key='ARCHER', n_clusters=n_ground_truth,
                   method=tool, use_pca=True, random_seed=seed)

        y_true = adata.obs['ground_truth'].astype(str)
        y_pred = adata.obs['ARCHER'].astype(str)
        ari = adjusted_rand_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ami = adjusted_mutual_info_score(y_true, y_pred)
        homogeneity = homogeneity_score(y_true, y_pred)
        v_measure = v_measure_score(y_true, y_pred)

        joint_feat = adata.obsm['ARCHER_emb']
        le = LabelEncoder()
        y_pred_int = le.fit_transform(y_pred)
        sil_score = silhouette_score(joint_feat, y_pred_int)

        print(f"Result -> ARI: {ari:.4f} | NMI: {nmi:.4f} | Silhouette: {sil_score:.4f}")
        res_dict = {
            'dataset': dataset_name, 'seed': seed, 'ARI': ari, 'NMI': nmi, 'AMI': ami,
            'Homogeneity': homogeneity, 'V-measure': v_measure, 'Silhouette': sil_score
        }
        dataset_results.append(res_dict)
        all_results.append(res_dict)

    df_ds = pd.DataFrame(dataset_results)
    df_ds.to_csv(f"results/ARCHER_{dataset_name}_results.csv", index=False)
    print("\n" + "=" * 80)
    print(f" SUMMARY FOR {dataset_name} ".center(80, "="))
    print("=" * 80)
    print(df_ds.describe().loc[['mean', 'std']])
    print("=" * 80)

# Final Overall Summary Table Across All Datasets
df_all = pd.DataFrame(all_results)
df_all.to_csv("results/ARCHER_all_results.csv", index=False)
metrics_cols = ['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-measure', 'Silhouette']

print("\n" + "=" * 95)
print(" FINAL BENCHMARK SUMMARY ACROSS ALL DATASETS (MEAN RESULTS) ".center(95, "="))
print("=" * 95)
mean_table = df_all.groupby('dataset')[metrics_cols].mean()
print(mean_table.round(4).to_string())

print("\n--- MEAN ± STD METRICS PER DATASET ---")
summary_rows = []
for ds_name, group in df_all.groupby('dataset'):
    row = {'dataset': ds_name}
    for m in metrics_cols:
        row[m] = f"{group[m].mean():.4f} ± {group[m].std():.4f}"
    summary_rows.append(row)

df_summary_fmt = pd.DataFrame(summary_rows)
print(df_summary_fmt.to_string(index=False))